<a href="https://colab.research.google.com/github/Krishdshah/HousePredictionModel/blob/main/HousePredictionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download the dataset
path = kagglehub.dataset_download("shree1992/housedata")

100%|██████████| 432k/432k [00:00<00:00, 66.9MB/s]

Extracting files...


In [8]:
import os
import pandas as pd

# The 'path' variable is set by kagglehub.dataset_download
# Based on the file listing, we will assume 'data.csv' is the training data
# and 'output.csv' is the test data (or for submission).

train_csv_path = os.path.join(path, 'data.csv')
test_csv_path = os.path.join(path, 'output.csv') # Assuming output.csv is the test set

# Check if the files exist before loading
if not os.path.exists(train_csv_path):
    print(f"Error: '{train_csv_path}' not found.")
if not os.path.exists(test_csv_path):
    print(f"Error: '{test_csv_path}' not found.")

# Load the dataframes
train_df = pd.read_csv(train_csv_path)
test_df = pd.read_csv(test_csv_path)

print("Train dataset loaded successfully from data.csv. First 5 rows:")
display(train_df.head())

print("\nTest dataset loaded successfully from output.csv. First 5 rows:")
display(test_df.head())

Train dataset loaded successfully from data.csv. First 5 rows:


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA



Test dataset loaded successfully from output.csv. First 5 rows:


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


In [13]:
# Identify categorical and numerical columns
# Exclude 'date' and 'price' from numerical_cols for initial identification, as 'date' needs special handling
# and 'price' is the target.
all_numerical_cols = train_df.select_dtypes(include=['int64', 'float64']).columns
numerical_cols = [col for col in all_numerical_cols if col not in ['price']]

all_categorical_cols = train_df.select_dtypes(include=['object']).columns
categorical_cols = [col for col in all_categorical_cols if col not in ['date']] # 'date' is a special categorical/temporal column

# Drop the 'date' column for now, as it requires specific temporal feature engineering not included in this simple preprocessing.
# If 'date' is important, it needs to be converted into numerical features (e.g., year, month, day, day_of_week).
if 'date' in train_df.columns:
    train_df = train_df.drop('date', axis=1)
if 'date' in test_df.columns:
    test_df = test_df.drop('date', axis=1)

# There is no 'Id' column in the provided dataframes (data.csv and output.csv).
# So, skip dropping 'Id' and extracting 'test_ids' from it directly.

# Handle missing numerical values with the mean
for col in numerical_cols: # This list already excludes 'price'
    train_df[col] = train_df[col].fillna(train_df[col].mean())
    test_df[col] = test_df[col].fillna(test_df[col].mean())

# Handle missing categorical values with the mode
for col in categorical_cols:
    train_df[col] = train_df[col].fillna(train_df[col].mode()[0])
    test_df[col] = test_df[col].fillna(test_df[col].mode()[0])

# One-hot encode categorical features
train_df = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=categorical_cols, drop_first=True)

# Align columns - very important for test set when one-hot encoding
# 'price' is the target variable
train_labels = train_df['price']
train_features = train_df.drop('price', axis=1)

# For the submission file, we need 'Id'. Since the datasets don't have it, we create sequential IDs for the test set.
test_ids = pd.Series(range(1, len(test_df) + 1), name='Id') # Create dummy IDs starting from 1

# If test_df still contains 'price' (which it does based on kernel state for output.csv),
# we need to drop it before making predictions, as 'price' is the target.
if 'price' in test_df.columns:
    test_df = test_df.drop('price', axis=1)

common_cols = list(set(train_features.columns) & set(test_df.columns))
train_features = train_features[common_cols]
test_df = test_df[common_cols]

# Ensure the order of columns is the same for both train_features and test_df
# This is crucial for models like RandomForest to work correctly.
sorted_common_cols = sorted(common_cols)
train_features = train_features[sorted_common_cols]
test_df = test_df[sorted_common_cols]


print("Data preprocessing complete. First 5 rows of processed train features:")
display(train_features.head())
print("\nFirst 5 rows of processed test features:")
display(test_df.head())

Data preprocessing complete. First 5 rows of processed train features:


,bathrooms,bedrooms,city_Auburn,city_Beaux Arts Village,city_Bellevue,city_Black Diamond,city_Bothell,city_Burien,city_Carnation,city_Clyde Hill,...,street_Schmitz Park to Alki Trail,street_Shangri-La Way NW,street_Sunrise Loop Trail,street_Tolt Pipeline Trail,street_Trossachs Blvd SE,street_Valley View Trail,view,waterfront,yr_built,yr_renovated
0,1.50,3.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,0,0,1955,2005
1,2.50,5.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,4,0,1921,0
2,2.00,3.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,0,0,1966,0
3,2.25,3.0,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,0,0,1963,0
4,2.50,4.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,0,0,1976,1992



First 5 rows of processed test features:


,bathrooms,bedrooms,city_Auburn,city_Beaux Arts Village,city_Bellevue,city_Black Diamond,city_Bothell,city_Burien,city_Carnation,city_Clyde Hill,...,street_Schmitz Park to Alki Trail,street_Shangri-La Way NW,street_Sunrise Loop Trail,street_Tolt Pipeline Trail,street_Trossachs Blvd SE,street_Valley View Trail,view,waterfront,yr_built,yr_renovated
0,1.50,3.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,0,0,1955,2005
1,2.50,5.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,4,0,1921,0
2,2.00,3.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,0,0,1966,0
3,2.25,3.0,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,0,0,1963,0
4,2.50,4.0,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,0,0,1976,1992


In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Split training data for validation
X_train, X_val, y_train, y_val = train_test_split(train_features, train_labels, test_size=0.2, random_state=42)

# Initialize and train the model
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


In [15]:
# Make predictions on the validation set
val_predictions = model.predict(X_val)

# Evaluate the model
mse = mean_squared_error(y_val, val_predictions)
rmse = mse**0.5
print(f"Root Mean Squared Error on validation set: {rmse}")

# Make predictions on the actual test set
test_predictions = model.predict(test_df)

print("Predictions on the test set are generated.")

Root Mean Squared Error on validation set: 984434.3392987942
Predictions on the test set are generated.


In [16]:
# Create submission DataFrame
submission_df = pd.DataFrame({'Id': test_ids, 'SalePrice': test_predictions})

# Display the first few rows of the submission file
print("Submission file head:")
display(submission_df.head())

# Save the submission file
submission_df.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")

Submission file head:


,Id,SalePrice
0,1,3.107816e+05
1,2,2.188911e+06
2,3,3.438186e+05
3,4,4.539337e+05
4,5,5.348140e+05


Submission file 'submission.csv' created successfully.
